# ✈️ Travel + Food AI Agent
Uses: **Gemini** · **Weatherstack** · **Wikipedia/Wikidata** · **Tavily** · **OpenStreetMap** (no API key needed)

Replace the 3 `"your_api_key"` values before running.

In [ ]:
!pip install google-genai requests tavily-python --quiet

In [ ]:
GEMINI_API_KEY       = "your_api_key"        # https://aistudio.google.com
WEATHERSTACK_API_KEY = "your_api_key"  # https://weatherstack.com
TAVILY_API_KEY       = "your_api_key"        # https://app.tavily.com


## 📦 Imports & clients

In [ ]:
import requests
from datetime import datetime
from urllib.parse import quote_plus
from google import genai
from tavily import TavilyClient

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options={"api_version": "v1"}
)

tavily = TavilyClient(api_key=TAVILY_API_KEY)

## 🌐 Tool 1 — Wikipedia summary

In [ ]:
def get_wikipedia_summary(place):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{place}"
    headers = {"User-Agent": "TravelAI/1.0"}
    res = requests.get(url, headers=headers)
    data = res.json()
    return data.get("extract", "No info found")

## 🗺️ Tool 2 — Wikidata: tourist places

In [ ]:
def get_places_from_wikidata(country="India"):
    url = "https://query.wikidata.org/sparql"
    query = """
    SELECT ?placeLabel WHERE {
      ?place wdt:P31 wd:Q570116;
             wdt:P17 wd:Q668.
      SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
    }
    LIMIT 5
    """
    headers = {"User-Agent": "TravelAI/1.0"}
    res = requests.get(url, params={"query": query, "format": "json"}, headers=headers)
    data = res.json()
    places = [item["placeLabel"]["value"] for item in data["results"]["bindings"]]
    return places

## ⛅ Tool 3 — Weatherstack: current weather & forecast

In [ ]:
def get_weather(city):
    """Fetch full weather details from Weatherstack API."""
    url    = "http://api.weatherstack.com/current"
    params = {"access_key": WEATHERSTACK_API_KEY, "query": city, "units": "m"}
    res    = requests.get(url, params=params)
    data   = res.json()

    if "current" not in data:
        err = data.get("error", {}).get("info", "Unknown error")
        return {"error": err}

    cur = data["current"]
    loc = data["location"]
    return {
        "temp"        : cur["temperature"],
        "feels_like"  : cur["feelslike"],
        "humidity"    : cur["humidity"],
        "pressure"    : cur["pressure"],
        "wind_speed"  : cur["wind_speed"],          # already km/h
        "wind_dir"    : cur["wind_dir"],
        "visibility"  : cur["visibility"],
        "weather"     : cur["weather_descriptions"][0] if cur["weather_descriptions"] else "N/A",
        "cloud_cover" : cur["cloudcover"],
        "uv_index"    : cur["uv_index"],
        "city"        : loc["name"],
        "country"     : loc["country"],
        "local_time"  : loc["localtime"],
    }


def get_weather_forecast(place):
    """Fetch forecast from Weatherstack (free tier returns current only; used for feasibility)."""
    url    = "http://api.weatherstack.com/current"
    params = {"access_key": WEATHERSTACK_API_KEY, "query": place, "units": "m"}
    res    = requests.get(url, params=params)
    data   = res.json()

    if "current" not in data:
        print("Weatherstack API Error:", data)
        return []

    cur = data["current"]
    loc = data["location"]
    # Return as a list with one entry so feasibility check works the same way
    return [{
        "date"      : loc["localtime"],
        "temp"      : cur["temperature"],
        "feels"     : cur["feelslike"],
        "humidity"  : cur["humidity"],
        "condition" : cur["weather_descriptions"][0] if cur["weather_descriptions"] else "N/A"
    }]


## 🗺️ Tool 4 — OpenStreetMap links (100% free, zero sign-up)

Uses **Nominatim** (OpenStreetMap's free geocoder) to find coordinates of a place,
then builds 3 clickable links:
- City map
- Restaurant search
- Overpass Turbo (shows all restaurants near the location on a live map)

In [ ]:
def get_map_links(place, food_query=None):
    headers = {"User-Agent": "TravelFoodAgent/1.0"}

    # Geocode the place using Nominatim (free, no key needed)
    geo_url = "https://nominatim.openstreetmap.org/search"
    params  = {"q": place, "format": "json", "limit": 1}
    res     = requests.get(geo_url, params=params, headers=headers)
    results = res.json()

    if not results:
        return {
            "map_link":        f"https://www.openstreetmap.org/search?query={quote_plus(place)}",
            "restaurant_link": f"https://www.openstreetmap.org/search?query={quote_plus('restaurants in ' + place)}",
            "overpass_link":   "https://overpass-turbo.eu",
            "lat": None, "lon": None
        }

    lat = float(results[0]["lat"])
    lon = float(results[0]["lon"])

    map_link        = f"https://www.openstreetmap.org/#map=13/{lat}/{lon}"
    restaurant_link = f"https://www.openstreetmap.org/search?query={quote_plus((food_query or 'restaurant') + ' in ' + place)}"
    overpass_link   = (
        f"https://overpass-turbo.eu/?Q=%5Bout%3Ajson%5D%5Btimeout%3A25%5D%3B%0A"
        f"node%5B%22amenity%22%3D%22restaurant%22%5D(around%3A2000%2C{lat}%2C{lon})%3B%0A"
        f"out+body%3B&R"
    )

    return {
        "map_link":        map_link,
        "restaurant_link": restaurant_link,
        "overpass_link":   overpass_link,
        "lat": lat, "lon": lon
    }

## 🍜 Tool 5 — Famous Food & Where-to-Eat Finder

Tavily searches the web → Gemini writes a food guide **with specific areas/neighbourhoods** → OpenStreetMap finds restaurants.


In [ ]:
def get_famous_food(place):
    print(f"\n🔍 Searching web for famous food + neighbourhoods in {place}...")

    # ── Tavily: famous dishes ─────────────────────────────────────────────
    dish_results = tavily.search(
        query=f"most famous traditional food dishes of {place}",
        max_results=5,
        search_depth="basic"
    )
    dish_snippets = "\n".join(
        f"- {r['title']}: {r['content'][:300]}"
        for r in dish_results.get("results", [])
    )

    # ── Tavily: areas / neighbourhoods known for food ─────────────────────
    area_results = tavily.search(
        query=f"best areas neighbourhoods streets to eat local food in {place}",
        max_results=5,
        search_depth="basic"
    )
    area_snippets = "\n".join(
        f"- {r['title']}: {r['content'][:300]}"
        for r in area_results.get("results", [])
    )

    # ── Gemini: food guide WITH area info ─────────────────────────────────
    prompt = f"""You are a world-class food travel guide.
Based on the web search results below, write a concise food guide for a traveller visiting {place}.

=== Famous Dishes Search Results ===
{dish_snippets}

=== Food Areas / Neighbourhoods Search Results ===
{area_snippets}

Your response MUST include ALL of the following sections:

1. 🍛 TOP 3 FAMOUS DISHES — For each dish:
   - Name & 1-2 sentence description
   - Specific area / neighbourhood / street in {place} where it is most famous or easiest to find
   - One iconic restaurant or food stall name in that area (if known)

2. 🗺️ TOP 3 FOOD AREAS / NEIGHBOURHOODS — Name and describe 3 areas in {place} that are
   famous for local food, street food markets, or food culture. For each area mention
   what type of food or vibe it is known for.

3. 🕐 BEST TIME TO EAT — Best time of day and best season to enjoy the local food scene.

4. 💰 PRICE GUIDE — Short tip on price range (budget / mid / upscale).

Keep the tone friendly and practical. Use emojis. Be specific about location names.
"""
    response   = client.models.generate_content(model="models/gemini-2.5-flash", contents=prompt)
    food_guide = response.text

    # ── OpenStreetMap links — no API key needed ────────────────────────────
    maps = get_map_links(place, food_query=f"famous food restaurant {place}")

    return food_guide, maps


## 🤖 Tool 6 — Gemini general Q&A

In [ ]:
def gemini_output(prompt_text):
    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=prompt_text
    )
    return response.text

## ⏰ Date helper & travel feasibility

In [ ]:
def is_far_future(date_str):
    user_date = datetime.strptime(date_str, "%Y-%m-%d")
    return (user_date - datetime.today()).days > 5


def check_travel_feasibility(place, date):
    if is_far_future(date):
        prompt = f"User wants to visit {place} on {date}. Based on typical seasonal weather, should they visit? Give clear advice."
    else:
        weather_data = get_weather_forecast(place)
        if not weather_data:
            return "⚠️ Weather data unavailable. Check city name or Weatherstack API key."
        prompt = f"User wants to visit {place} on {date}. Current weather: {weather_data}. Should they visit? Give clear advice."

    response = client.models.generate_content(model="models/gemini-2.5-flash", contents=prompt)
    return response.text


## 🧠 Main Agent — intent router

In [ ]:
def travel_agent(user_query):
    q = user_query.lower()

    # ── Food intent ───────────────────────────────────────────────────────────
    if any(w in q for w in ["food", "eat", "dish", "cuisine", "local food", "where to eat"]):
        place = user_query
        for kw in ["food in", "eat in", "where to eat in", "cuisine of", "dishes of", "dishes in", "in "]:
            if kw in q:
                place = user_query[q.index(kw) + len(kw):].strip()
                break

        food_guide, maps = get_famous_food(place)
        print("\n" + "="*60)
        print(f"🍽️  FAMOUS FOOD OF {place.upper()} — WITH AREAS & WHERE TO EAT")
        print("="*60)
        print(food_guide)
        print("\n📍 Find these spots on the map (no sign-in needed):")
        print(f"   🗺️  City map           : {maps['map_link']}")
        print(f"   🍴  Find restaurants   : {maps['restaurant_link']}")
        print(f"   🔎  All nearby on map  : {maps['overpass_link']}")

        # If the user also mentions weather/temperature in the same query, show temp
        if any(w in q for w in ["weather", "temperature", "temp", "hot", "cold", "climate"]):
            # extract city from food place
            w = get_weather(place)
            if "error" not in w:
                print(f"\n🌡️  Current Temperature in {w['city']}: {w['temp']}°C  ({w['weather']})")
        return

    # ── Tourist places intent ─────────────────────────────────────────────────
    if any(w in q for w in ["places", "visit", "tourist", "sightseeing"]):
        places = get_places_from_wikidata()
        result = [f"📍 {p}:\n{get_wikipedia_summary(p)}" for p in places]
        print("\n\n".join(result))
        return

    # ── Weather intent — show ONLY temperature (and condition) ────────────────
    if any(w in q for w in ["weather", "temperature", "temp", "how hot", "how cold"]):
        city = user_query
        for kw in ["weather in", "weather of", "temperature in", "temp in"]:
            if kw in q:
                city = user_query[q.index(kw) + len(kw):].strip() or "Delhi"
                break
        w = get_weather(city)
        if "error" in w:
            print(f"⚠️ Could not fetch weather: {w['error']}")
        else:
            print(f"\n🌡️  Temperature in {w['city']}, {w['country']}: {w['temp']}°C  ({w['weather']})")
        return

    # ── Fallback: Gemini ──────────────────────────────────────────────────────
    print(gemini_output(user_query))


## 🏃 Full decision agent — place + date + food + map all at once

In [ ]:
def travel_decision_agent():
    place = input("🌍 Where do you want to visit? ")
    date  = input("📅 When do you plan to visit? (YYYY-MM-DD) ")

    # ── Places ───────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"📸 PLACES TO VISIT IN {place.upper()}")
    print("="*60)
    print(gemini_output(f"Top places to visit in {place}. Keep it concise."))

    # ── Food ─────────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"🍽️  FAMOUS FOOD OF {place.upper()} — WITH AREAS & WHERE TO EAT")
    print("="*60)
    food_guide, maps = get_famous_food(place)
    print(food_guide)
    print("\n📍 Find these spots on the map (no sign-in needed):")
    print(f"   🗺️  City map           : {maps['map_link']}")
    print(f"   🍴  Find restaurants   : {maps['restaurant_link']}")
    print(f"   🔎  All nearby on map  : {maps['overpass_link']}")

    # ── Full Weather Card (Weatherstack) ──────────────────────────────────────
    print("\n" + "="*50)
    print(f"⛅  CURRENT WEATHER IN {place.upper()}")
    print("="*50)
    w = get_weather(place)
    if "error" in w:
        print(f"⚠️  Could not fetch weather: {w['error']}")
    else:
        print(f"🌡️  Temperature   : {w['temp']}°C")
        print(f"🤔  Feels Like    : {w['feels_like']}°C")
        print(f"🌤️  Condition     : {w['weather']}")
        print(f"💧  Humidity      : {w['humidity']}%")
        print(f"🌬️  Wind Speed    : {w['wind_speed']} km/h  ({w['wind_dir']})")
        print(f"👁️  Visibility    : {w['visibility']} km")
        print(f"☁️  Cloud Cover   : {w['cloud_cover']}%")
        print(f"🔆  UV Index      : {w['uv_index']}")
        print(f"📊  Pressure      : {w['pressure']} mb")
        print(f"🕐  Local Time    : {w['local_time']}")
        print(f"📍  Location      : {w['city']}, {w['country']}")
    print("="*50)

    # ── Travel Feasibility ───────────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"🌤️  TRAVEL FEASIBILITY — {place.upper()} on {date}")
    print("="*60)
    print(check_travel_feasibility(place, date))


In [ ]:
print("🤖 Travel & Food AI Agent")
print("Ask me about places, food, or weather. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye! Safe travels! ✈️")
        break
    if not user_input:
        continue
    travel_agent(user_input)
    print()

🤖 Travel & Food AI Agent
Ask me about places, food, or weather. Type 'quit' to exit.

You: famous food agra

🔍 Searching web for famous food + neighbourhoods in famous food agra...

🍽️  FAMOUS FOOD OF FAMOUS FOOD AGRA — WITH AREAS & WHERE TO EAT
Welcome to Agra, a city not just of iconic monuments but also of a vibrant culinary soul! Prepare your taste buds for a delicious journey blending royal Mughlai traditions with humble street food delights. Here's your concise guide to eating like a local:

---

### 🍛 TOP 3 FAMOUS DISHES

1.  **Chaat (Bhalla, Kachori, Samosas, Gol Gappas, Tikki)** 🌶️
    *   **Description:** Agra's chaats are a symphony of spicy, tangy, and sweet flavors. From crispy *bhalla* and stuffed *kachori* to the explosive *gol gappas* and flavorful *tikki*, these savory snacks are a street food staple.
    *   **Area:** **Chaat Gali, Sadar Bazaar** is the heart of Agra's chaat scene, famous for its decades-old vendors.
    *   **Iconic Stall:** Look for **Sardar Ji Pane

In [ ]:
travel_decision_agent()

🌍 Where do you want to visit? agra
📅 When do you plan to visit? (YYYY-MM-DD) 2025-09-09

📸 PLACES TO VISIT IN AGRA
Here are the top places to visit in Agra:

*   **Taj Mahal:** The iconic marble mausoleum, a UNESCO World Heritage site.
*   **Agra Fort:** A massive red sandstone fort, also a UNESCO site.
*   **Fatehpur Sikri:** A historic imperial city, located slightly outside Agra.
*   **Itmad-ud-Daulah's Tomb (Baby Taj):** An elegant Mughal mausoleum, considered a precursor to the Taj.
*   **Mehtab Bagh:** A charbagh complex offering stunning sunset views of the Taj Mahal.
*   **Akbar's Tomb, Sikandra:** The tomb of the great Mughal emperor Akbar.

🍽️  FAMOUS FOOD OF AGRA — WITH AREAS & WHERE TO EAT

🔍 Searching web for famous food + neighbourhoods in agra...
Namaste, fellow food adventurer! 🍛 Get ready to discover that Agra is far more than just the Taj Mahal – it's a culinary treasure trove where royal Mughlai flavors meet vibrant street food. Here's your concise guide to eating li